# 📋 Módulo 02 · DataFrames

Desarrollar intuición sobre la estructura de una tabla: índice, columnas como Series, selección, ejes, qué devuelve cada operación y cómo se relacionan las tablas.

Versión web con conceptos, reconocimiento, razonamiento y proyecto: [https://rosalesluciano.github.io/datalab/](https://rosalesluciano.github.io/datalab/#/m/m02)

## Cómo trabajar este notebook
1. Ejecutá la celda ⚙️ (carga datos y verificador).
2. **Antes de escribir código**, escribí en un comentario cómo lo vas a resolver.
3. Implementá y guardá tu respuesta en `resultado` (o lo que pida el ejercicio).
4. `comprobar()` verifica. Si te trabás: `ver_pista("id", 1)`, después 2 y 3.
5. `ver_solucion("id")` es el último recurso: muestra idea → pseudocódigo → código.

## Conceptos del módulo
- **Estructura: filas, columnas e índice**: Entender qué estás mirando: un DataFrame es una tabla con **índice** (etiquetas de filas), **columnas** con nombre y **valores**.
- **Cada columna es una Series**: Saber qué tipo de objeto obtenés al seleccionar columnas, porque cambia qué métodos podés usar después.
- **El índice**: Usar etiquetas de fila con significado (producto, fecha, cliente) para buscar y alinear datos.
- **Tipos de las columnas**: Cada columna tiene **un** tipo, que define qué operaciones son posibles (sumar, `.str`, `.dt`).
- **Seleccionar con loc e iloc**: Seleccionar filas **y** columnas a la vez, por etiqueta (`loc`) o por posición (`iloc`).
- **Máscaras booleanas por dentro**: Entender que un filtro es una **Series de True/False** alineada con las filas, que se puede guardar, combinar, contar y reutilizar.
- **Crear, modificar y eliminar columnas**: Derivar variables nuevas (importe, margen, tramo) y ajustar la tabla (renombrar, descartar).
- **Transformar valores**: Convertir valores en otros: traducir códigos, clasificar en tramos, aplicar reglas.
- **Ejes: axis=0 y axis=1**: Elegir la dirección de una operación: resumir cada columna o resumir cada fila.
- **Qué devuelve cada operación**: Predecir la **forma** del resultado: ¿una fila por grupo o una por registro?, ¿Series o DataFrame?
- **Reestructurar: formato ancho y largo**: Cambiar la forma de la tabla: de 'una columna por trimestre' (ancho) a 'una fila por tienda y trimestre' (largo), o al revés.
- **Relaciones entre DataFrames**: Entender cómo se relacionan las tablas (un cliente tiene muchos pedidos) para combinarlas sin perder ni multiplicar filas.
- **DataFrames y SQL**: Traer datos de una base de datos a un DataFrame (y guardar resultados), combinando lo mejor de SQL y pandas.

In [ ]:
#@title ⚙️ 1) Ejecutá esta celda primero (datos y verificador) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series", list: "una lista", dict: "un diccionario",
               tuple: "una tupla", set: "un set", str: "un texto (str)", int: "un entero (int)",
               float: "un decimal (float)", bool: "un booleano (bool)", type(None): "None (nada)",
               np.ndarray: "un array de NumPy"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    if isinstance(e, dict):
        return (isinstance(u, dict) and set(map(str, u)) == set(map(str, e))
                and all(_scalar_eq(u[k] if k in u else u[str(k)], v) for k, v in e.items()))
    if isinstance(e, (set, frozenset)):
        return isinstance(u, (set, frozenset)) and u == e
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    if u is None:
        return False, ("Tu resultado es None. ¿Te falta un `return` en la función, o guardaste lo que devuelve "
                       "un método que modifica en el lugar (como `.sort()` o `.append()`)?")
    contenedores = (list, tuple, dict, set, str)
    if isinstance(e, contenedores) and not isinstance(u, type(e)) and not (
            isinstance(e, (list, tuple)) and isinstance(u, (list, tuple))):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}."
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple)) and len(u) != len(e):
        return False, f"Tu resultado tiene {len(u)} elementos y se esperaban {len(e)}."
    if isinstance(e, dict) and isinstance(u, dict) and set(map(str, u)) != set(map(str, e)):
        faltan = [k for k in e if str(k) not in set(map(str, u))]
        sobran = [k for k in u if str(k) not in set(map(str, e))]
        msg = "Las claves del diccionario no coinciden."
        if faltan:
            msg += f" Faltan: {faltan}."
        if sobran:
            msg += f" Sobran: {sobran}."
        return False, msg
    txt = repr(u)
    return False, f"Tu resultado es {txt[:300] + ('…' if len(txt) > 300 else '')}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usá & (y), | (o) y poné cada condición entre paréntesis."),
    ("can't multiply sequence", "Estás multiplicando un texto (str). ¿El número llegó como texto? Convertilo con int() o float()."),
    ("can only concatenate str", "Estás sumando texto con otra cosa. Convertí con str() para unir textos, o con float() para hacer cuentas."),
    ("unsupported operand type", "Los dos lados de la operación tienen tipos incompatibles (por ejemplo, int + str o None + float). Revisá type() de cada valor."),
    ("could not convert string to float", "Hay un texto que no representa un número (por ejemplo 'n/d' o ''). Filtralo o capturá el ValueError con try/except."),
    ("invalid literal for int", "Hay un texto que no representa un entero. Revisá el valor o usá try/except ValueError."),
    ("index out of range", "Pediste una posición que no existe. Recordá: las posiciones van de 0 a len()-1, y -1 es el último."),
    ("values to unpack", "La cantidad de variables a la izquierda no coincide con la cantidad de valores que llegan."),
    ("ZeroDivisionError", "Dividiste por cero. ¿La lista o el divisor pueden estar vacíos? Verificalo antes de dividir."),
    ("'NoneType' object", "Un valor es None. Suele pasar con una función sin return o con métodos como .sort() que devuelven None."),
    ("KeyError", "Esa clave o columna no existe. Revisá mayúsculas, tildes y espacios; con dicts podés usar .get()."),
    ("NameError", "Usás una variable que no existe (todavía). ¿Error de tipeo o la definiste más abajo?"),
    ("IndentationError", "Revisá la sangría: el contenido de if, for, def, etc. va con 4 espacios."),
    ("SyntaxError", "Error de sintaxis: revisá paréntesis, comillas, dos puntos (:) al final de if/for/def y que cada línea esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este tipo de objeto. Revisá el nombre y qué tipo tiene la variable."),
    ("MergeError", "Revisá las columnas que usás para unir (on / left_on / right_on)."),
    ("TypeError", "Algún tipo de dato no encaja con la operación. Revisá type() de las variables involucradas."),
    ("ValueError", "El tipo es correcto pero el valor no es válido para esta operación."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\nimport sqlite3\nconn = sqlite3.connect(":memory:")\nclientes.to_sql("clientes", conn, index=False)\nproductos.to_sql("productos", conn, index=False)\npedidos.to_sql("pedidos", conn, index=False)\n\ntrimestral = pd.DataFrame({\n    "tienda": ["Centro", "Norte", "Sur"],\n    "Q1": [100, 80, 60],\n    "Q2": [120, 95, 70],\n    "Q3": [130, 90, 85],\n})\n'
_SOL_B64 = '...'
_TABLAS = ['clientes', 'productos', 'pedidos', 'empleados', 'ventas_diarias', 'sucio', 'trimestral', 'conn (SQLite)']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todos los datos limpios (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    from IPython.display import HTML, display
    import html as _h
    if ok:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">✅ Verificado. Ahora explicá con tus palabras por qué funciona '
                     'y probá una variante.</div>'))
    else:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_pista(ej, nivel=1):
    """Pistas progresivas (1-3). Pedí la siguiente solo si la anterior no alcanzó."""
    pistas = _SOL[ej]["hints"]
    print(f"Pista {nivel}/{len(pistas)}: {pistas[min(nivel, len(pistas)) - 1]}")

def ver_solucion(ej):
    """Último recurso: primero la idea, después el pseudocódigo y al final el código."""
    info = _SOL[ej]
    print("💡 Idea:\n" + info["explain"] + "\n")
    print("🧭 Pseudocódigo:\n" + info["pseudo"] + "\n")
    print("🧑‍💻 Código:\n" + info["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 Anatomía de un DataFrame

## 1. Tu primer DataFrame
`Código` · `Muy fácil`

Creá un DataFrame con la columna `jugador` (`Messi, Ronaldo, Mbappé`) y la columna `goles` (`30, 25, 28`), en ese orden.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = pd.DataFrame({
    # 'columna': [valores],
})

comprobar("pd12")

## 2. Conoce tus tablas
`Código` · `Muy fácil`

¿Qué tamaño tiene la tabla `clientes` (filas y columnas)? Guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd13")

## 3. Inventario de columnas
`Código` · `Muy fácil`

Guardá en `resultado` una **lista** con los nombres de las columnas de `pedidos`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("pd15")

---
# 📌 Columnas: cada una es una Series

## 4. Una o varias columnas
`Código` · `Fácil`

Para una lista de contactos necesitás solo el `nombre` y la `ciudad` de cada cliente, en ese orden. Guardá el DataFrame en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df01")

## 5. Dos columnas, un corchete
`Debugging` · `Muy fácil`

Este código intenta quedarse con las columnas `producto` y `precio`, pero falla. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

resultado = productos['producto', 'precio']

comprobar("df-e01")

---
# 📌 Índice y selección (loc / iloc)

## 6. Etiqueta o posición
`Código` · `Medio`

Con la Series `ventas`, guardá en `resultado` una **lista** con dos valores: las ventas del **Mié** (buscando por etiqueta) y las del **primer día** (buscando por posición).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

ventas = pd.Series({'Lun': 120, 'Mar': 95, 'Mié': 143, 'Jue': 88, 'Vie': 170})
resultado = [ , ]

comprobar("pd03")

## 7. loc: filas y columnas por etiqueta
`Código` · `Medio`

De `productos`, tomá las filas con índice **2 a 4 (incluido)** y solo las columnas `producto` y `precio`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = productos.loc[   ]

comprobar("df02")

## 8. iloc: filas por posición
`Código` · `Medio`

Guardá en `resultado` los **3 últimos clientes** de la tabla (por posición).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df03")

## 9. Buscar un valor concreto
`Código` · `Medio`

¿Cuál es el `precio` del producto **Monitor**? Guardá el número (no una tabla) en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df04")

## 10. Ficha de dos productos
`Código` · `Medio`

Armá una tabla con el `precio` y el `stock` de **Laptop** y **Tablet**, con el nombre del producto como índice. Guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df-d01")

## 11. iloc con una etiqueta
`Debugging` · `Fácil`

Este código quiere el precio del Monitor con el producto como índice, pero falla. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

tabla = productos.set_index('producto')
resultado = tabla.iloc['Monitor', 'precio']

comprobar("df-e02")

---
# 📌 Máscaras booleanas por dentro

## 12. ¿Qué proporción no tiene stock?
`Código` · `Fácil`

¿Qué **proporción** de productos tiene stock en 0? Guardá el número (entre 0 y 1) en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df-d02")

---
# 📌 Crear, modificar y eliminar columnas

## 13. Columnas calculadas
`Código` · `Fácil`

¿Cuánto dinero hay inmovilizado en inventario? Agregá a `productos` la columna `valor_stock` = `precio × stock` y guardá el DataFrame completo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

# crea la columna y luego:
resultado = productos

comprobar("df11")

## 14. Transferencia: precio con IVA y margen
`Transferencia` · `Medio`

Agregá a `productos` la columna `precio_iva` (precio × 1.21) y la columna `iva` (precio_iva − precio). Guardá en `resultado` las columnas `producto`, `precio`, `precio_iva` e `iva`.

> 🔁 **Qué cambia:** Antes creaste una columna con una operación entre dos columnas. Ahora vas a crear **dos** columnas, y una depende de la otra.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df-g01")

## 15. Renombrar y descartar
`Código` · `Medio`

Preparando un reporte de `pedidos`: eliminá la columna `estado` y renombrá `cantidad` como `unidades`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df13")

## 16. El descuento que no se aplicó
`Debugging` · `Medio`

Querías aplicar un **10% de descuento** solo a los productos de la categoría Accesorios, pero los precios siguen igual. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

productos[productos['categoria'] == 'Accesorios']['precio'] = productos['precio'] * 0.9
resultado = productos

comprobar("df-e03")

## 17. Categorías por tramos
`Código` · `Difícil`

Clasificá a los clientes en tramos de edad: **joven** (hasta 29), **adulto** (30 a 44) y **senior** (45 o más), en una columna nueva `rango_edad`. Guardá solo `nombre`, `edad` y `rango_edad`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

clientes['rango_edad'] = 
resultado = 

comprobar("df12")

## 18. Traducir valores con map
`Código` · `Medio`

Traducí los talles de `talles`: `S` → `Chico`, `M` → `Mediano`, `L` → `Grande`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

talles = pd.Series(['S', 'M', 'L', 'M', 'S'])
resultado = 

comprobar("pd09")

## 19. Tu propia función con apply
`Código` · `Difícil`

Clasificá cada valor de `temperaturas` como **frío** (menor que 20), **templado** (entre 20 y 25, ambos incluidos) o **calor** (mayor que 25), usando una función propia aplicada a la Series.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

temperaturas = pd.Series([18.5, 22.0, 26.3, 20.0, 25.0, 31.2])

def clasificar(t):
    # completa aquí
    pass

resultado = temperaturas.apply(clasificar)

comprobar("pd10")

---
# 📌 Ejes: filas vs columnas

## 20. Total anual por tienda
`Código` · `Medio`

A partir de `trimestral`, calculá el **total de ventas de cada tienda** (Q1 + Q2 + Q3), con el nombre de la tienda como índice. Guardá la Series en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

print(trimestral)

resultado = 

comprobar("df-d03")

## 21. Transferencia: promedio de cada trimestre
`Transferencia` · `Medio`

Calculá el **promedio de ventas de cada trimestre** (Q1, Q2, Q3) entre las tiendas. Guardá la Series en `resultado`.

> 🔁 **Qué cambia:** Antes resumiste **cada fila** (tienda). Ahora querés resumir **cada columna** (trimestre).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df-g02")

---
# 📌 Qué devuelve cada operación

## 22. Comparar con tu grupo
`Código` · `Desafío`

¿Qué empleados cobran **más que la media de su propio departamento**? Guardalos en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

media_depto = empleados.groupby('depto')['salario'].transform('mean')
resultado = 

comprobar("df21")

## 23. La columna llena de NaN
`Debugging` · `Medio`

Querías agregar a cada empleado la media salarial de su departamento, pero la columna nueva sale llena de NaN. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

empleados['media_depto'] = empleados.groupby('depto')['salario'].mean()
resultado = empleados[['nombre', 'depto', 'salario', 'media_depto']]

comprobar("df-e04")

## 24. Ranking por grupo
`Código` · `Desafío`

Agregá a `empleados` la columna `ranking` con la posición del salario **dentro de su depto** (1 = el que más cobra; empates con método `dense`). Guardá `nombre`, `depto`, `salario` y `ranking`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

empleados['ranking'] = 
resultado = empleados[['nombre', 'depto', 'salario', 'ranking']]

comprobar("df30")

---
# 📌 Reestructurar: ancho y largo

## 25. Tabla dinámica
`Código` · `Difícil`

Armá una tabla cruzada con `estado` en las filas, `categoria` en las columnas y la **suma de `cantidad`** (los huecos deben ser 0).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

df = pedidos.merge(productos, on='producto_id')
resultado = pd.pivot_table(df, )

comprobar("df26")

## 26. De ancho a largo con melt
`Código` · `Difícil`

Pasá `trimestral` a formato **largo** con las columnas `tienda`, `trimestre` y `ventas`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = trimestral.melt()

comprobar("df27")

---
# 📌 Relaciones entre DataFrames

## 27. Pedidos por cliente (incluidos los 0)
`Código` · `Difícil`

Calculá **cuántos pedidos** hizo cada cliente (por `nombre`), **incluyendo a los que no hicieron ninguno** (con 0). Guardá la Series en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df-d04")

---
# 📌 DataFrames y bases de datos SQL

## 28. Consultar una base de datos
`Código` · `Medio`

Desde la base de datos `conn`, traé el `nombre` y la `ciudad` de los clientes con **edad mayor a 40**, usando SQL.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = pd.read_sql('''
    SELECT ...
''', conn)

comprobar("df31")

## 29. Parámetros seguros
`Código` · `Difícil`

El usuario elige un precio mínimo (`precio_min`). Traé los productos con `precio` mayor a ese valor, **pasándolo como parámetro** (no pegado en el texto de la consulta).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

precio_min = 100
resultado = pd.read_sql('SELECT * FROM productos WHERE precio > ?', conn, )

comprobar("df32")

## 30. Guardar en la base de datos
`Código` · `Difícil`

Calculá con pandas el número de clientes por país (columnas `pais` y `n_clientes`), guardalo en la tabla `resumen_pais` de la base y leelo de vuelta con SQL en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resumen = 
resumen.to_sql('resumen_pais', conn, index=False, if_exists='replace')
resultado = pd.read_sql('SELECT * FROM resumen_pais', conn)

comprobar("df33")

## 31. La IA traduce SQL a pandas
`IA copiloto` · `Medio`

Tenías esta consulta SQL y le pediste a una IA el equivalente en pandas:

```sql
SELECT segmento, AVG(edad) AS edad_media
FROM clientes
WHERE pais <> 'España'
GROUP BY segmento
ORDER BY edad_media DESC;
```

> 🤖 Prompt usado:
> Traducí esta consulta SQL a pandas y explicame qué parte del código corresponde a cada cláusula (WHERE, GROUP BY, AVG, ORDER BY).

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

resultado = (clientes[clientes['pais'] != 'España']
             .groupby('segmento')['edad'].mean()
             .sort_values(ascending=False))

comprobar("df-f01")

---
# 📌 Proyecto: radiografía de clientes

## 32. Fase 2 · Construir la tabla
`Código` · `Difícil`

Construí la tabla resumen diseñada: columnas `cliente_id`, `nombre`, `segmento`, `n_pedidos`, `facturacion`; una fila por cliente (12), con 0 para quienes no compraron. Sin pedidos cancelados.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("df-p2")

## 33. Fase 3 · Comparar segmentos
`Código` · `Medio`

Con la tabla resumen (ya construida en el código inicial), calculá la **facturación promedio por cliente** de cada `segmento`, ordenada de mayor a menor.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

ped = pedidos[pedidos['estado'] != 'cancelado'].merge(productos, on='producto_id')
ped['importe'] = ped['cantidad'] * ped['precio']
por_cliente = ped.groupby('cliente_id').agg(n_pedidos=('pedido_id', 'count'), facturacion=('importe', 'sum')).reset_index()
resumen = (clientes[['cliente_id', 'nombre', 'segmento']]
           .merge(por_cliente, on='cliente_id', how='left')
           .fillna({'n_pedidos': 0, 'facturacion': 0}))

resultado = 

comprobar("df-p3")

## 34. Fase 4 · El promedio inflado
`Debugging` · `Difícil`

Una versión alternativa calcula el promedio por segmento con un **inner join** y da números más altos. Corregila para que considere a todos los clientes.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

ped = pedidos[pedidos['estado'] != 'cancelado'].merge(productos, on='producto_id')
ped['importe'] = ped['cantidad'] * ped['precio']
por_cliente = ped.groupby('cliente_id')['importe'].sum().reset_index(name='facturacion')
resumen = clientes.merge(por_cliente, on='cliente_id')
resultado = resumen.groupby('segmento')['facturacion'].mean().sort_values(ascending=False)

comprobar("df-p4")

---
# 🏁 Fin del notebook
Volvé a la [plataforma](https://rosalesluciano.github.io/datalab/) para los ejercicios de reconocimiento, razonamiento y el modo **¿Sé esto?**.